In [1]:
# Cell 1: Imports, config, seeds, utilities

import os, random, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageFilter

import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import h5py

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, cohen_kappa_score, roc_curve, auc,
                             precision_recall_fscore_support, balanced_accuracy_score,
                             precision_recall_curve)
from sklearn.preprocessing import label_binarize

# Optional XAI libs
gradcam_available = False
lime_available = False
try:
    from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image
    gradcam_available = True
except Exception as e:
    print("pytorch-grad-cam not available:", e)
try:
    from lime import lime_image
    lime_available = True
except Exception as e:
    print("lime not available:", e)

# ---------------- CONFIG ----------------
DATA_DIR = r"F:\Sorted Data set"   # folder per class
EXPECTED_NUM_CLASSES = 9
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0
SEED = 42
NUM_EPOCHS = 20
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = True if DEVICE.type == 'cuda' else False
SAVE_DIR = './runs_ct_all_models_live_h5'
os.makedirs(SAVE_DIR, exist_ok=True)

UPLOADED_IMAGE = r"F:\Sorted Data set\Brain Tumor\ct_tumor (2).png"  # for LIME demo (optional)

# seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p

def list_images_by_topfolder(data_dir):
    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise RuntimeError(f"DATA_DIR not found: {data_dir}")
    exts = ('.png', '.jpg', '.jpeg', '.tiff', '.bmp')
    rows = []
    for folder in [p for p in data_dir.iterdir() if p.is_dir()]:
        lbl = folder.name.strip()
        for p in folder.rglob('*'):
            if p.is_file() and p.suffix.lower() in exts:
                rows.append({'path': str(p), 'label': lbl})
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No images found in {data_dir}.")
    df['label'] = df['label'].astype(str).str.strip()
    return df

def apply_clahe(img_gray):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(img_gray)

def add_gaussian_noise(np_img, mean=0, var=5):
    sigma = var ** 0.5
    gauss = np.random.normal(mean, sigma, np_img.shape).astype(np.float32)
    noisy = np_img.astype(np.float32) + gauss
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)
    return noisy

def edge_enhance_pil(img_pil):
    return img_pil.filter(ImageFilter.EDGE_ENHANCE_MORE)


C:\Users\tabib\anaconda3\envs\tf-gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 2: Augmentations, dataset, loaders, preprocessing visualization

train_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.08, rotate_limit=12, p=0.5),
    A.CLAHE(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(var_limit=(5.0,40.0), p=0.25),
    A.RandomGamma(p=0.25),
    A.OneOf([A.Sharpen(), A.Emboss()], p=0.15),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
])

class CTFolderDataset(Dataset):
    def __init__(self, df, label2idx=None, transform=None, gray=True, extra_preproc=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.gray = gray
        self.extra_preproc = extra_preproc
        if label2idx is None:
            labels_sorted = sorted(self.df['label'].unique())
            self.label2idx = {l:i for i,l in enumerate(labels_sorted)}
        else:
            self.label2idx = label2idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row['path']
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE if self.gray else cv2.IMREAD_COLOR)
        if img is None:
            img = Image.open(path).convert('L' if self.gray else 'RGB')
            img = np.array(img)
        if self.extra_preproc:
            if random.random() < 0.25:
                try: img = apply_clahe(img)
                except Exception: pass
            if random.random() < 0.15:
                try:
                    pil = Image.fromarray(img); pil = edge_enhance_pil(pil); img = np.array(pil)
                except Exception: pass
            if random.random() < 0.15:
                try: img = add_gaussian_noise(img, var=random.uniform(1,20))
                except Exception: pass
        if self.transform:
            if self.gray:
                img = np.expand_dims(img, axis=-1)
            augmented = self.transform(image=img)
            img_t = augmented['image']
        else:
            img = cv2.resize(img, (IMG_SIZE,IMG_SIZE))
            if self.gray: img = np.expand_dims(img, axis=-1)
            img_t = TF.to_tensor(img)
        label = self.label2idx[row['label']]
        return img_t, label

def make_loaders_from_df(df, test_size=0.2, val_size=0.1):
    train_val, test = train_test_split(df, test_size=test_size, stratify=df['label'], random_state=SEED)
    train, val = train_test_split(train_val, test_size=val_size/(1-test_size), stratify=train_val['label'], random_state=SEED)
    labels_sorted = sorted(train['label'].unique())
    label2idx = {l:i for i,l in enumerate(labels_sorted)}
    train_ds = CTFolderDataset(train, label2idx=label2idx, transform=train_aug, extra_preproc=True)
    val_ds = CTFolderDataset(val, label2idx=label2idx, transform=val_transform, extra_preproc=False)
    test_ds = CTFolderDataset(test, label2idx=label2idx, transform=val_transform, extra_preproc=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds

def visualize_preprocessing(df, outdir, n=8, gray=True):
    outdir = ensure_dir(outdir)
    sample_df = df.sample(n=min(n, len(df)), random_state=SEED)
    fig, axes = plt.subplots(len(sample_df), 2, figsize=(8, 3*len(sample_df)))
    for i, (_, row) in enumerate(sample_df.iterrows()):
        img = cv2.imread(row['path'], cv2.IMREAD_GRAYSCALE if gray else cv2.IMREAD_COLOR)
        if img is None:
            img = np.array(Image.open(row['path']).convert('L' if gray else 'RGB'))
        disp_before = img
        img_after = apply_clahe(img)
        pil = Image.fromarray(img_after); pil = edge_enhance_pil(pil); img_after = np.array(pil)
        img_after = cv2.resize(img_after, (IMG_SIZE, IMG_SIZE))
        axes[i,0].imshow(disp_before, cmap='gray' if gray else None); axes[i,0].set_title(f"Before ({row['label']})"); axes[i,0].axis('off')
        axes[i,1].imshow(img_after, cmap='gray' if gray else None); axes[i,1].set_title("After (CLAHE+Edge+Resize)"); axes[i,1].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, 'preprocessing_before_after.png')); plt.close()
    print(f"Saved preprocessing visualization to {outdir}")


In [3]:
# Cell 3: StrongCNN and advanced CNNs

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample:
            identity = self.downsample(identity)
        out = self.relu(out + identity)
        return out

class StrongCNN(nn.Module):
    def __init__(self, in_ch=1, num_classes=2, dropout=0.5):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)
        )
        self.layer1 = nn.Sequential(ResidualBlock(64, 64, 1), ResidualBlock(64, 64, 1))
        self.layer2 = nn.Sequential(ResidualBlock(64, 128, 2), ResidualBlock(128, 128, 1))
        self.layer3 = nn.Sequential(ResidualBlock(128, 256, 2), ResidualBlock(256, 256, 1))
        self.layer4 = nn.Sequential(ResidualBlock(256, 512, 2))
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout*0.8),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = self.avgpool(x)
        x = self.classifier(x)
        return x

class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, ch//r), nn.ReLU(inplace=True),
            nn.Linear(ch//r, ch), nn.Sigmoid()
        )
    def forward(self, x):
        w = self.fc(x)
        w = w.view(x.size(0), x.size(1), 1, 1)
        return x * w

class AdvancedCNN1(nn.Module):
    def __init__(self, in_ch=1, num_classes=2):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), SEBlock(128), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), SEBlock(256), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes))
    def forward(self, x): return self.fc(self.body(x))

class AdvancedCNN2(nn.Module):
    def __init__(self, in_ch=1, num_classes=2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 32, 5, stride=2, padding=2), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.block = nn.Sequential(
            ResidualBlock(64, 64, 1),
            ResidualBlock(64, 128, 2),
            ResidualBlock(128, 128, 1),
            ResidualBlock(128, 256, 2)
        )
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(256, num_classes))
    def forward(self, x): return self.fc(self.avg(self.block(self.stem(x))))

class AdvancedCNN3(nn.Module):
    def __init__(self, in_ch=1, num_classes=2):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, 48, 3, padding=1), nn.GroupNorm(6, 48), nn.ReLU(),
            nn.Conv2d(48, 96, 3, padding=1), nn.GroupNorm(12, 96), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(96, 192, 3, padding=1), nn.GroupNorm(24, 192), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(192, 384, 3, padding=1), nn.GroupNorm(48, 384), nn.ReLU(), nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.6), nn.Linear(384, 192), nn.ReLU(), nn.Dropout(0.4), nn.Linear(192, num_classes))
    def forward(self, x): return self.fc(self.body(x))

class AdvancedCNN4(nn.Module):
    def __init__(self, in_ch=1, num_classes=2):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, 64, 7, stride=2, padding=3), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(3, stride=2, padding=1),
            ResidualBlock(64, 128, 2), ResidualBlock(128, 128, 1),
            ResidualBlock(128, 256, 2), SEBlock(256),
            ResidualBlock(256, 512, 2)
        )
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, num_classes))
    def forward(self, x): return self.fc(self.avg(self.body(x)))

class AdvancedCNN5(nn.Module):
    def __init__(self, in_ch=1, num_classes=2):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(256, num_classes))
    def forward(self, x): return self.fc(self.body(x))

class AdvancedCNN6(nn.Module):
    def __init__(self, in_ch=1, num_classes=2):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            ResidualBlock(64, 64, 1),
            nn.MaxPool2d(2),
            ResidualBlock(64, 128, 2),
            nn.MaxPool2d(2),
            ResidualBlock(128, 256, 2),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.55), nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.35), nn.Linear(128, num_classes))
    def forward(self, x): return self.fc(self.body(x))

class AdvancedCNN7(nn.Module):
    def __init__(self, in_ch=1, num_classes=2):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), SEBlock(64),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), SEBlock(128),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), SEBlock(256),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, num_classes))
    def forward(self, x): return self.fc(self.body(x))


In [4]:
# Cell 4: timm builders (ViT + 7 transfer learning models)

def build_vit(name='vit_base_patch16_224', pretrained=True, in_ch=1, num_classes=2):
    model = timm.create_model(name, pretrained=pretrained, num_classes=num_classes, in_chans=in_ch)
    return model

def build_transfer_model(name, pretrained=True, in_ch=1, num_classes=2, freeze_ratio=0.6):
    model = timm.create_model(name, pretrained=pretrained, num_classes=num_classes, in_chans=in_ch)
    # Partial freezing to reduce overfitting
    total = sum(1 for _ in model.parameters())
    freeze_until = int(total * freeze_ratio)
    idx = 0
    for p in model.parameters():
        if idx < freeze_until:
            p.requires_grad = False
        idx += 1
    return model

TL_MODELS = [
    'resnet50',
    'densenet121',
    'efficientnet_b0',
    'mobilenetv3_large_100',
    'convnext_tiny',
    'swin_tiny_patch4_window7_224',
    'seresnet50'
]


In [5]:
# Cell 5: Training routines (tqdm per-batch + TensorBoard per-epoch), validation, history plots

class LabelSmoothingCE(nn.Module):
    def __init__(self, eps=0.1):
        super().__init__()
        self.eps = eps
    def forward(self, logits, targets):
        n_classes = logits.size(1)
        log_probs = F.log_softmax(logits, dim=1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.eps / (n_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1 - self.eps)
        return torch.mean(torch.sum(-true_dist * log_probs, dim=1))

def train_one_epoch(model, loader, criterion, optimizer, device, grad_clip=1.0):
    model.train()
    running_loss = 0.0
    all_preds, all_targets = [], []
    for xb, yb in tqdm(loader, desc="Train", leave=False):
        xb = xb.to(device, dtype=torch.float)
        yb = yb.to(device, dtype=torch.long)
        optimizer.zero_grad(set_to_none=True)
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        preds = torch.argmax(out, dim=1).detach().cpu().numpy()
        all_preds.extend(preds.tolist())
        all_targets.extend(yb.detach().cpu().numpy().tolist())
    epoch_loss = running_loss / len(loader.dataset)
    acc = (np.array(all_preds) == np.array(all_targets)).mean()
    kappa = cohen_kappa_score(all_targets, all_preds) if len(set(all_targets))>1 else 0.0
    return epoch_loss, acc, kappa

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_targets, all_probs = [], [], []
    for xb, yb in tqdm(loader, desc="Val", leave=False):
        xb = xb.to(device, dtype=torch.float)
        yb = yb.to(device, dtype=torch.long)
        out = model(xb)
        loss = criterion(out, yb)
        running_loss += loss.item() * xb.size(0)
        probs = F.softmax(out, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_preds.extend(preds.tolist())
        all_targets.extend(yb.cpu().numpy().tolist())
        all_probs.extend(probs.tolist())
    epoch_loss = running_loss / len(loader.dataset)
    acc = (np.array(all_preds) == np.array(all_targets)).mean()
    kappa = cohen_kappa_score(all_targets, all_preds) if len(set(all_targets))>1 else 0.0
    return epoch_loss, acc, kappa, np.array(all_targets), np.array(all_preds), np.array(all_probs)

def plot_history(history, name, outdir):
    ensure_dir(outdir)
    plt.figure(); plt.plot(history['train_loss'], label='train'); plt.plot(history['val_loss'], label='val'); plt.legend(); plt.title(name + ' loss'); plt.savefig(os.path.join(outdir, f'{name}_loss.png')); plt.close()
    plt.figure(); plt.plot(history['train_acc'], label='train'); plt.plot(history['val_acc'], label='val'); plt.legend(); plt.title(name + ' accuracy'); plt.savefig(os.path.join(outdir, f'{name}_accuracy.png')); plt.close()
    plt.figure(); plt.plot(history['train_kappa'], label='train'); plt.plot(history['val_kappa'], label='val'); plt.legend(); plt.title(name + ' kappa'); plt.savefig(os.path.join(outdir, f'{name}_kappa.png')); plt.close()

def fit_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS, lr=1e-4, wd=1e-4, save_name='model', patience=5):
    model = model.to(DEVICE)
    criterion = LabelSmoothingCE(eps=0.1)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5)
    history = {'train_loss':[], 'train_acc':[], 'train_kappa':[], 'val_loss':[], 'val_acc':[], 'val_kappa':[]}
    best_val_loss = 1e9
    best_state = None
    no_improve = 0
    model_outdir = ensure_dir(os.path.join(SAVE_DIR, save_name))
    writer = SummaryWriter(log_dir=os.path.join(model_outdir, 'tb'))

    for epoch in range(num_epochs):
        tr_loss, tr_acc, tr_kappa = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc, val_kappa, y_true, y_pred, y_probs = validate_one_epoch(model, val_loader, criterion, DEVICE)
        scheduler.step(val_loss)
        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc); history['train_kappa'].append(tr_kappa)
        history['val_loss'].append(val_loss); history['val_acc'].append(val_acc); history['val_kappa'].append(val_kappa)

        # TensorBoard logging (live)
        writer.add_scalar('Loss/train', tr_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        writer.add_scalar('Acc/train', tr_acc, epoch)
        writer.add_scalar('Acc/val', val_acc, epoch)
        writer.add_scalar('Kappa/train', tr_kappa, epoch)
        writer.add_scalar('Kappa/val', val_kappa, epoch)

        print(f"[{save_name}] Epoch {epoch+1}/{num_epochs} - train_loss={tr_loss:.4f} acc={tr_acc:.4f} kappa={tr_kappa:.4f} | val_loss={val_loss:.4f} acc={val_acc:.4f} kappa={val_kappa:.4f}")

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {k: v.cpu() if isinstance(v, torch.Tensor) else v for k, v in model.state_dict().items()}
            torch.save(model.state_dict(), os.path.join(model_outdir, f'{save_name}_best.pt'))
            no_improve = 0
        else:
            no_improve += 1
        pd.DataFrame(history).to_csv(os.path.join(model_outdir, f'{save_name}_history.csv'), index=False)
        plot_history(history, save_name, model_outdir)
        if optimizer.param_groups[0]['lr'] < 1e-7 or no_improve >= patience:
            print('Early stop.')
            break

    writer.close()
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


In [6]:
# Cell 6: Evaluation helpers (confusion, ROC, PR/recall, F1, kappa), CV curves

def plot_confusion_matrix(cm, classes, outpath, title='Confusion matrix'):
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.ylabel('True'); plt.xlabel('Pred'); plt.title(title)
    plt.tight_layout(); plt.savefig(outpath); plt.close()

def plot_roc_multiclass(y_true, y_probs, n_classes, classes, outpath_prefix):
    y_bin = label_binarize(y_true, classes=list(range(n_classes)))
    plt.figure(figsize=(10,8)); fprs={}; tprs={}; aucs={}
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
        roc_auc = auc(fpr, tpr); fprs[i]=fpr; tprs[i]=tpr; aucs[i]=roc_auc
        plt.plot(fpr, tpr, lw=1, label=f'{classes[i]} AUC={roc_auc:.2f}')
    all_fpr = np.unique(np.concatenate([fprs[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fprs[i], tprs[i])
    mean_tpr /= n_classes
    macro_auc = auc(all_fpr, mean_tpr)
    plt.plot(all_fpr, mean_tpr, color='k', lw=2, linestyle='--', label=f'macro AUC={macro_auc:.2f}')
    plt.plot([0,1],[0,1], color='navy', lw=1, linestyle=':')
    plt.xlim([0.0,1.0]); plt.ylim([0.0,1.05]); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC curves'); plt.legend(loc='lower right')
    plt.savefig(outpath_prefix + '_roc.png'); plt.close()
    return macro_auc

def plot_pr_curves(y_true, y_probs, n_classes, classes, outpath_prefix):
    y_bin = label_binarize(y_true, classes=list(range(n_classes)))
    plt.figure(figsize=(10,8))
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_bin[:, i], y_probs[:, i])
        pr_auc = auc(recall, precision)
        plt.plot(recall, precision, lw=1, label=f'{classes[i]} AP={pr_auc:.2f}')
    plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall curves')
    plt.legend(loc='lower left'); plt.savefig(outpath_prefix + '_pr.png'); plt.close()

def evaluate_on_loader(model, loader, device):
    model.eval()
    y_true=[]; y_pred=[]; y_probs=[]
    with torch.no_grad():
        for xb,yb in loader:
            xb = xb.to(device, dtype=torch.float)
            out = model(xb)
            probs = F.softmax(out, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            y_true.extend(yb.numpy().tolist())
            y_pred.extend(preds.tolist())
            y_probs.extend(probs.tolist())
    y_true = np.array(y_true); y_pred = np.array(y_pred); y_probs = np.array(y_probs)
    cm = confusion_matrix(y_true, y_pred)
    rpt = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    prf = precision_recall_fscore_support(y_true, y_pred, zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred) if len(set(y_true))>1 else 0.0
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    auc_val = None
    try:
        n_classes = len(np.unique(y_true))
        if y_probs.size and y_probs.shape[1] >= n_classes:
            y_bin = label_binarize(y_true, classes=np.unique(y_true))
            auc_val = roc_auc_score(y_bin, y_probs[:, :n_classes], average='macro', multi_class='ovr')
    except Exception:
        auc_val = None
    return {'cm':cm, 'report':rpt, 'prf':prf, 'kappa':kappa, 'auc':auc_val, 'balanced_accuracy':bal_acc, 'y_true':y_true, 'y_pred':y_pred, 'y_probs':y_probs}

def save_full_evaluation(model, name, test_loader, train_ds, outdir):
    ensure_dir(outdir)
    metrics = evaluate_on_loader(model, test_loader, DEVICE)
    classes = [k for k,v in sorted(train_ds.label2idx.items(), key=lambda x:x[1])]
    plot_confusion_matrix(metrics['cm'], classes, os.path.join(outdir, f'{name}_cm.png'), title=f'{name} Confusion Matrix')
    report_df = pd.DataFrame(metrics['report']).transpose()
    report_df.to_csv(os.path.join(outdir, f'{name}_classification_report.csv'))
    prec, rec, f1, sup = metrics['prf']
    prf_df = pd.DataFrame({'precision': prec, 'recall': rec, 'f1': f1, 'support': sup})
    prf_df.index = classes
    prf_df.to_csv(os.path.join(outdir, f'{name}_prf_by_class.csv'))
    pred_counts = Counter(metrics['y_pred'])
    with open(os.path.join(outdir, f'{name}_pred_counts.txt'),'w') as f: f.write(str(pred_counts))
    with open(os.path.join(outdir, f'{name}_kappa.txt'),'w') as f: f.write(str(metrics['kappa']))
    with open(os.path.join(outdir, f'{name}_balanced_accuracy.txt'),'w') as f: f.write(str(metrics.get('balanced_accuracy','NA')))
    n_classes = len(classes)
    if metrics['y_probs'].size != 0 and metrics['y_probs'].shape[1] >= n_classes:
        macro_auc = plot_roc_multiclass(metrics['y_true'], metrics['y_probs'][:, :n_classes], n_classes, classes, os.path.join(outdir, f'{name}'))
        plot_pr_curves(metrics['y_true'], metrics['y_probs'][:, :n_classes], n_classes, classes, os.path.join(outdir, f'{name}'))
        with open(os.path.join(outdir, f'{name}_auc.txt'),'w') as f: f.write(str(macro_auc))
    print(f"{name} evaluation saved to {outdir}")
    return metrics


In [7]:
# Cell 7: XAI functions (Grad-CAM, Grad-CAM++, LIME)

def get_target_layer_for_model(model):
    for nm, mod in reversed(list(model.named_modules())):
        if isinstance(mod, nn.Conv2d):
            return mod
    return None

def run_and_save_gradcam_variants(model, loader, device, outdir, classes, num_images=6, use_pp=False):
    if not gradcam_available:
        print("Grad-CAM not installed; skipping.")
        return
    ensure_dir(outdir)
    model = model.to(device).eval()
    target_layer = get_target_layer_for_model(model)
    if target_layer is None:
        print("No suitable Conv2d found for Grad-CAM; skipping.")
        return
    cam_cls = GradCAMPlusPlus if use_pp else GradCAM
    # ✅ updated line for your version
    cam = cam_cls(model=model, target_layers=[target_layer])

    saved = 0
    for xb, yb in loader:
        if saved >= num_images: break
        for i in range(xb.size(0)):
            if saved >= num_images: break
            img_t = xb[i].unsqueeze(0).to(device, dtype=torch.float)
            true_label = int(yb[i].item())
            preds = model(img_t)
            pred_label = int(torch.argmax(preds, dim=1).item())
            try:
                grayscale_cam = cam(input_tensor=img_t, targets=[ClassifierOutputTarget(pred_label)])[0]
            except Exception as e:
                print("GradCAM forward error:", e); continue
            np_img = img_t.squeeze().cpu().numpy()
            if np_img.ndim == 3:
                disp = np.transpose(np_img, (1,2,0))
            else:
                disp = np_img
            disp = (disp - disp.min())/(disp.max()-disp.min()+1e-9)
            if disp.ndim == 2:
                disp = np.expand_dims(disp, axis=-1)
            vis = show_cam_on_image(disp, grayscale_cam, use_rgb=False)
            outpath = os.path.join(outdir, f'{"gradcampp" if use_pp else "gradcam"}_{saved}_pred{classes[pred_label]}_true{classes[true_label]}.png')
            cv2.imwrite(outpath, cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
            saved += 1
    print(f"Saved {saved} {'Grad-CAM++' if use_pp else 'Grad-CAM'} images to {outdir}")



def run_and_save_lime(model, pil_transform_fn, pil_images, device, outdir, num_samples=300, classes=None):
    ensure_dir(outdir)
    if not lime_available:
        print("LIME not installed; skipping.")
        return
    model = model.to(device).eval()
    explainer = lime_image.LimeImageExplainer()
    def batch_predict(images):
        batch = []
        for im in images:
            pil = Image.fromarray(im.astype('uint8'))
            t = pil_transform_fn(pil)
            arr = t.numpy() if isinstance(t, torch.Tensor) else np.array(t)
            batch.append(arr)
        batch = np.stack(batch, axis=0)
        batch = torch.tensor(batch, dtype=torch.float).to(device)
        with torch.no_grad():
            out = model(batch)
            probs = F.softmax(out, dim=1).cpu().numpy()
        return probs
    for idx, pil_img in enumerate(pil_images):
        try:
            np_img = np.array(pil_img.convert('RGB'))
            explanation = explainer.explain_instance(np_img, batch_predict, top_labels=3, hide_color=0, num_samples=num_samples)
            pred_label = np.argmax(batch_predict([np_img])[0])
            temp, mask = explanation.get_image_and_mask(pred_label, positive_only=False, num_features=10, hide_rest=False)
            outpath = os.path.join(outdir, f'lime_{idx}_pred{classes[pred_label] if classes else pred_label}.png')
            cv2.imwrite(outpath, cv2.cvtColor(temp, cv2.COLOR_RGB2BGR))
        except Exception as e:
            print("LIME failed for image", idx, ":", e)
    print("Saved LIME explanations to", outdir)


In [8]:
# Cell 8: Save PyTorch model weights to H5 using h5py

def save_model_state_dict_to_h5(model, h5_path):
    """
    Save PyTorch state_dict tensors to an H5 file.
    Note: This stores tensors as datasets under their state_dict keys.
    """
    state = model.state_dict()
    with h5py.File(h5_path, 'w') as h5f:
        for k, v in state.items():
            h5f.create_dataset(k, data=v.cpu().numpy())
    print(f"Saved model weights to H5: {h5_path}")

def load_model_state_dict_from_h5(model, h5_path, strict=True):
    """
    Load tensors from H5 back into a PyTorch model's state_dict.
    """
    with h5py.File(h5_path, 'r') as h5f:
        state = {}
        for k in h5f.keys():
            state[k] = torch.tensor(h5f[k][()])
    model.load_state_dict(state, strict=strict)
    print(f"Loaded model weights from H5: {h5_path}")


In [9]:
# Cell 9: Per-model evaluation & XAI; ensemble and MC dropout

def evaluate_and_save(model, name, test_loader, train_ds, model_outdir):
    ensure_dir(model_outdir)
    metrics = save_full_evaluation(model, name, test_loader, train_ds, model_outdir)
    classes = [k for k,v in sorted(train_ds.label2idx.items(), key=lambda x:x[1])]
    run_and_save_gradcam_variants(model, test_loader, DEVICE, os.path.join(model_outdir, 'gradcam'), classes, num_images=6, use_pp=False)
    run_and_save_gradcam_variants(model, test_loader, DEVICE, os.path.join(model_outdir, 'gradcampp'), classes, num_images=6, use_pp=True)
    try:
        pil_demo = Image.open(UPLOADED_IMAGE).convert('L')
        def pil_transform_fn(pil_img):
            pil_img = pil_img.resize((IMG_SIZE, IMG_SIZE))
            arr = np.array(pil_img).astype(np.float32)/255.0
            arr = (arr - 0.5)/0.5
            arr = np.expand_dims(arr, axis=0)  # 1,H,W
            return torch.tensor(arr, dtype=torch.float)
        run_and_save_lime(model, pil_transform_fn, [pil_demo], DEVICE, os.path.join(model_outdir, 'lime'), num_samples=300, classes=classes)
    except Exception as e:
        print("Skipping LIME demo:", e)
    return metrics

def mc_dropout_predict(model, loader, device, mc_runs=6):
    model.train()
    all_probs = []
    for _ in range(mc_runs):
        probs = []
        with torch.no_grad():
            for xb, _ in loader:
                xb = xb.to(device, dtype=torch.float)
                probs.append(F.softmax(model(xb), dim=1).cpu().numpy())
        all_probs.append(np.vstack(probs))
    all_probs = np.stack(all_probs, axis=0)
    mean_p = all_probs.mean(axis=0)
    entropy = -np.sum(mean_p * np.log(mean_p + 1e-9), axis=1)
    return mean_p, entropy

def evaluate_ensemble_and_save(models_list, names_list, test_loader, train_ds, ensemble_name='ensemble_all'):
    for m in models_list: m.eval()
    all_probs = []; y_true = []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE, dtype=torch.float)
            probs_sum = None
            for model in models_list:
                out = model(xb)
                probs = F.softmax(out, dim=1).cpu().numpy()
                probs_sum = probs if probs_sum is None else probs_sum + probs
            mean_probs = probs_sum / len(models_list)
            all_probs.append(mean_probs)
            y_true.extend(yb.numpy().tolist())
    all_probs = np.vstack(all_probs); y_true = np.array(y_true); y_pred = np.argmax(all_probs, axis=1)
    cm = confusion_matrix(y_true, y_pred)
    classes = [k for k,v in sorted(train_ds.label2idx.items(), key=lambda x:x[1])]
    outdir = ensure_dir(os.path.join(SAVE_DIR, ensemble_name))
    plot_confusion_matrix(cm, classes, os.path.join(outdir, f'{ensemble_name}_cm.png'), title=f'{ensemble_name} Confusion Matrix')
    rpt = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    pd.DataFrame(rpt).transpose().to_csv(os.path.join(outdir, f'{ensemble_name}_classification_report.csv'))
    with open(os.path.join(outdir, f'{ensemble_name}_pred_counts.txt'),'w') as f:
        f.write(str(Counter(y_pred)))
    try:
        plot_roc_multiclass(y_true, all_probs, len(classes), classes, os.path.join(outdir, ensemble_name))
        plot_pr_curves(y_true, all_probs, len(classes), classes, os.path.join(outdir, ensemble_name))
    except Exception as e:
        print("Ensemble ROC/PR failed:", e)
    entropies = []
    for model in models_list:
        _, entropy = mc_dropout_predict(model, test_loader, DEVICE, mc_runs=6)
        entropies.append(entropy)
    if len(entropies) > 0:
        entropies = np.stack(entropies, axis=0)
        avg_entropy = entropies.mean(axis=0)
        np.savetxt(os.path.join(outdir, f'{ensemble_name}_entropy.csv'), avg_entropy, delimiter=',')
        plt.figure(figsize=(8,4)); plt.plot(np.arange(len(avg_entropy)), avg_entropy, marker='.', linestyle='-')
        plt.xlabel('sample'); plt.ylabel('entropy'); plt.title(f'{ensemble_name} avg entropy'); plt.savefig(os.path.join(outdir, f'{ensemble_name}_entropy.png')); plt.close()
    print(f"Ensemble saved to {outdir}")


In [10]:
# Cell 10: Main — train all models (live), evaluate, XAI, save H5, ensemble

if __name__ == '__main__':
    print('Device:', DEVICE)
    df = list_images_by_topfolder(DATA_DIR)
    n_found = df['label'].nunique()
    print(f'Found {len(df)} images across {n_found} classes.')
    if n_found != EXPECTED_NUM_CLASSES:
        print(f"WARNING: expected {EXPECTED_NUM_CLASSES} classes but detected {n_found}. Proceeding with detected classes.")
    print(df['label'].value_counts())

    visualize_preprocessing(df, os.path.join(SAVE_DIR, 'preproc_vis'), n=8, gray=True)
    train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = make_loaders_from_df(df)
    num_classes = len(train_ds.label2idx)
    in_ch = 1

    # Models: StrongCNN, ViT, 7 advanced CNNs
    models_cfg = [
        ('strong_cnn', lambda in_ch, num_classes: StrongCNN(in_ch=in_ch, num_classes=num_classes, dropout=0.5), {'lr':1e-4, 'wd':1e-4}),
        ('vit_base',   lambda in_ch, num_classes: build_vit('vit_base_patch16_224', pretrained=True, in_ch=in_ch, num_classes=num_classes), {'lr':3e-5, 'wd':1e-4}),
        ('adv_cnn1',   lambda in_ch, num_classes: AdvancedCNN1(in_ch=in_ch, num_classes=num_classes), {'lr':1e-4, 'wd':2e-4}),
        ('adv_cnn2',   lambda in_ch, num_classes: AdvancedCNN2(in_ch=in_ch, num_classes=num_classes), {'lr':1.5e-4, 'wd':2e-4}),
        ('adv_cnn3',   lambda in_ch, num_classes: AdvancedCNN3(in_ch=in_ch, num_classes=num_classes), {'lr':1e-4, 'wd':3e-4}),
        ('adv_cnn4',   lambda in_ch, num_classes: AdvancedCNN4(in_ch=in_ch, num_classes=num_classes), {'lr':1e-4, 'wd':2e-4}),
        ('adv_cnn5',   lambda in_ch, num_classes: AdvancedCNN5(in_ch=in_ch, num_classes=num_classes), {'lr':1.2e-4, 'wd':2e-4}),
        ('adv_cnn6',   lambda in_ch, num_classes: AdvancedCNN6(in_ch=in_ch, num_classes=num_classes), {'lr':1e-4, 'wd':2e-4}),
        ('adv_cnn7',   lambda in_ch, num_classes: AdvancedCNN7(in_ch=in_ch, num_classes=num_classes), {'lr':1.1e-4, 'wd':2e-4}),
    ]

    # Add 7 transfer learning models
    for tl_name in TL_MODELS:
        models_cfg.append((
            f'tl_{tl_name}',
            lambda in_ch, num_classes, tl_name=tl_name: build_transfer_model(tl_name, pretrained=True, in_ch=in_ch, num_classes=num_classes, freeze_ratio=0.6),
            {'lr':8e-5, 'wd':2e-4}
        ))

    trained_models = []
    for name, builder, hparams in models_cfg:
        print(f"\nTraining {name} ...")
        model = builder(in_ch, num_classes)
        model, hist = fit_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS, lr=hparams['lr'], wd=hparams['wd'], save_name=name, patience=5)
        outdir = os.path.join(SAVE_DIR, name)
        metrics = evaluate_and_save(model, name, test_loader, train_ds, outdir)

        # Save model weights to H5
        h5_path = os.path.join(outdir, f'{name}_weights.h5')
        save_model_state_dict_to_h5(model, h5_path)

        trained_models.append((name, model))

    # Optional CV curves for one TL model
    try:
        def tl_resnet50(in_ch, num_classes): return build_transfer_model('resnet50', pretrained=True, in_ch=in_ch, num_classes=num_classes, freeze_ratio=0.6)
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        labels = df['label'].values
        macro_accs = []; kappas = []
        fold = 0
        for train_idx, val_idx in skf.split(np.zeros(len(labels)), labels):
            fold += 1
            train_df = df.iloc[train_idx]; val_df = df.iloc[val_idx]
            labels_sorted = sorted(train_df['label'].unique())
            label2idx = {l:i for i,l in enumerate(labels_sorted)}
            train_ds_cv = CTFolderDataset(train_df, label2idx=label2idx, transform=train_aug)
            val_ds_cv = CTFolderDataset(val_df, label2idx=label2idx, transform=val_transform)
            train_loader_cv = DataLoader(train_ds_cv, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
            val_loader_cv = DataLoader(val_ds_cv, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
            model_cv = tl_resnet50(in_ch=1, num_classes=len(label2idx))
            model_cv, _ = fit_model(model_cv, train_loader_cv, val_loader_cv, num_epochs=6, lr=8e-5, wd=2e-4, save_name=f'tl_resnet50_cv_fold{fold}', patience=3)
            metrics_cv = evaluate_on_loader(model_cv, val_loader_cv, DEVICE)
            macro_accs.append((np.array(metrics_cv['y_pred']) == np.array(metrics_cv['y_true'])).mean())
            kappas.append(metrics_cv['kappa'])
        plt.figure(); plt.plot(range(1, 6), macro_accs, marker='o'); plt.title('tl_resnet50 CV accuracy'); plt.xlabel('Fold'); plt.ylabel('Accuracy'); plt.savefig(os.path.join(SAVE_DIR, 'tl_resnet50_cv_acc.png')); plt.close()
        plt.figure(); plt.plot(range(1, 6), kappas, marker='o'); plt.title('tl_resnet50 CV kappa'); plt.xlabel('Fold'); plt.ylabel('Kappa'); plt.savefig(os.path.join(SAVE_DIR, 'tl_resnet50_cv_kappa.png')); plt.close()
        print("Saved CV curves for tl_resnet50")
    except Exception as e:
        print("CV run skipped:", e)

    print("\nRunning ensemble evaluation across all models...")
    evaluate_ensemble_and_save([m for _, m in trained_models], [n for n, _ in trained_models], test_loader, train_ds, ensemble_name='ensemble_all_models')

    print("\nAll done. Results saved under", SAVE_DIR)


Device: cuda
Found 12804 images across 9 classes.
label
Brain Tumor             2318
Brain Healthy           2300
Kidney Cyst             1800
Kidney Normal           1800
Kidney Stone            1716
Kidney Tumor            1340
Lung normal              631
Lung Malignant cases     561
Lung adenocarcinoma      338
Name: count, dtype: int64
Saved preprocessing visualization to ./runs_ct_all_models_live_h5\preproc_vis

Training strong_cnn ...


[strong_cnn] Epoch 1/20 - train_loss=1.2526 acc=0.6523 kappa=0.5935 | val_loss=0.9147 acc=0.8392 kappa=0.8133


[strong_cnn] Epoch 2/20 - train_loss=0.9365 acc=0.8208 kappa=0.7917 | val_loss=0.7863 acc=0.8876 kappa=0.8692


[strong_cnn] Epoch 3/20 - train_loss=0.8306 acc=0.8773 kappa=0.8575 | val_loss=0.6843 acc=0.9383 kappa=0.9284


[strong_cnn] Epoch 4/20 - train_loss=0.7824 acc=0.9018 kappa=0.8860 | val_loss=0.6674 acc=0.9360 kappa=0.9257


...

[tl_resnet50_cv_fold5] Epoch 2/6 - train_loss=0.8026 acc=0.8793 kappa=0.8598 | val_loss=0.6571 acc=0.9520 kappa=0.9442


[tl_resnet50_cv_fold5] Epoch 3/6 - train_loss=0.7085 acc=0.9314 kappa=0.9203 | val_loss=0.6269 acc=0.9586 kappa=0.9519


[tl_resnet50_cv_fold5] Epoch 4/6 - train_loss=0.6658 acc=0.9474 kappa=0.9389 | val_loss=0.6122 acc=0.9680 kappa=0.9628


[tl_resnet50_cv_fold5] Epoch 5/6 - train_loss=0.6409 acc=0.9576 kappa=0.9508 | val_loss=0.5939 acc=0.9766 kappa=0.9728


[tl_resnet50_cv_fold5] Epoch 6/6 - train_loss=0.6273 acc=0.9639 kappa=0.9581 | val_loss=0.5880 acc=0.9770 kappa=0.9732
Saved CV curves for tl_resnet50

Running ensemble evaluation across all models...
Ensemble saved to ./runs_ct_all_models_live_h5\ensemble_all_models

All done. Results saved under ./runs_ct_all_models_live_h5
